<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2010%20-%202026-05-06%20-%20Visualizaciones%20avanzadas/clase_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 10 · Visualizaciones avanzadas · 🧪 Taller

**Fecha:** Miércoles 6 de mayo de 2026 · **Duración estimada:** ~60 min · **Nivel:** Intermedio

> 📖 **La teoría completa vive en el [HTML de la sesión](./index.html).** Este cuaderno es el **taller práctico**: aquí codificas.

## 🎯 Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

1. **Construir** visualizaciones multivariadas con `pairplot`, `FacetGrid` y `jointplot` (Seaborn).
2. **Diseñar** gráficos interactivos con Plotly Express (`scatter`, `bar`, `histogram`).
3. **Evaluar** qué gráfico es adecuado para cada pregunta analítica.
4. **Analizar** relaciones bivariadas y multivariadas con código limpio.

## ✅ Prerrequisitos

- Haber completado Día 8-9: Pandas, matplotlib básico, estadística descriptiva.
- Conocer `groupby()`, `pivot_table()`, filtrado con `loc[]` e `iloc[]`.

## 🧭 Cómo usar este cuaderno

- Ejecuta cada celda con `Shift + Enter`.
- Las celdas 🟢🟡🔴 son ejercicios (verde → rojo = dificultad creciente).
- Las celdas `# %% [solution]` contienen la solución oculta: descomenta si estás atascado.
- Hay `assert` al final de cada ejercicio: si no falla, está correcto ✓.

---

# Setup

In [ ]:
# --- Setup del entorno ---------------------------------------------------
# Imports estándar, reproducibilidad y confirmación visual
# -----------------------------------------------------------------------

# Instalación (descomentar solo la primera vez en Colab)
# !pip install --quiet pandas numpy matplotlib seaborn plotly

# Stdlib
import random
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Estilo visual coherente en todo el curso
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")

---

## 🎬 Hook · Descubriendo patrones visuales en el Titanic

> **Pregunta de hoy:** Los datos del Titanic tienen muchas dimensiones. ¿Cómo podemos visualizar de qué manera la **edad, tarifa, clase y sexo** influyen en la supervivencia? ¿Un gráfico de barras alcanza? ¿Necesitamos interactividad?

Corre la celda siguiente y explora el resultado. ¿Qué patrones ves?

In [ ]:
# Cargar datos del Titanic (dataset público)
titanic = sns.load_dataset("titanic")

print(f"Shape: {titanic.shape}")
print(f"\nNulls:\n{titanic.isnull().sum()}")
print(f"\nColumnas: {titanic.columns.tolist()}")
titanic.head()

⏸️ **Pause and think:** ¿Cuántas variables numéricas vs categóricas ves? ¿Cuál sería un buen gráfico para ver cómo sobrevivencia vs edad varía por clase?

---

## 1. Pairplot · La matriz de scatter plots

⏱️ ~10 min · Demo del instructor + 5 min práctica

> 📌 **Concepto clave:** Un `pairplot` crea una matriz de gráficos: cada par de variables numéricas en un scatter, y la diagonal muestra la distribución univariada (histograma o densidad). Es ideal para explorar correlaciones y patrones visuales rápidamente.

In [ ]:
# Demo: pairplot simple del Titanic
# (seleccionamos solo algunas columnas para que sea legible)

vars_numericas = ["age", "fare", "survived"]
df_clean = titanic[vars_numericas].dropna()

# pairplot colorea por una variable categórica si queremos
g = sns.pairplot(
    df_clean,
    hue="survived",  # colorea por supervivencia (0=no, 1=sí)
    diag_kind="hist",  # diagonal = histogramas
    plot_kws={"alpha": 0.6},  # transparencia para ver solapamientos
    height=2.5
)
g.fig.suptitle("Pairplot: Edad, Tarifa y Supervivencia en el Titanic", y=1.01, fontsize=12)
plt.tight_layout()

💡 **Tip:** Los scatters están coloreados: rojo = murió, azul = sobrevivió. ¿Ves algún patrón claro?

### 🟢 Ejercicio 1 — Guiado · Pairplot con penguins

⏱️ ~5 min · **Patrón:** fill-in-the-blank

Crea un pairplot del dataset `penguins` usando las columnas `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` y coloreando por `species` (especie). Usa `diag_kind="kde"` en vez de histograma.

**Output esperado:** Una matriz 3x3 con los pares de variables y la diagonal en densidad.

In [ ]:
# 🟢 Completa los huecos marcados con ___

penguins = sns.load_dataset("penguins")

columnas_interes = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "species"]
df_peng = penguins[columnas_interes].dropna()

pp = sns.pairplot(
    df_peng,
    hue="___",  # ← color por especie
    diag_kind="___",  # ← usa densidad
    height=2
)
pp.fig.suptitle("Pairplot: Medidas de pingüinos", y=1.01)
plt.tight_layout()

In [ ]:
# Celda de validación — NO modificar
assert "pp" in dir(), "La variable pp no existe"
assert pp.fig is not None, "El pairplot no se creó correctamente"
print("✅ Ejercicio 1 completado")

In [ ]:
# %% [solution] — descomenta para ver la solución
# pp = sns.pairplot(
#     df_peng,
#     hue="species",
#     diag_kind="kde",
#     height=2
# )
# pp.fig.suptitle("Pairplot: Medidas de pingüinos", y=1.01)
# plt.tight_layout()

---

## 2. FacetGrid · Subgráficos por grupo

⏱️ ~10 min · Práctica individual

> 📌 **Concepto clave:** Un `FacetGrid` divide el gráfico en múltiples paneles (facetas), cada una mostrando un subconjunto de datos según una o más variables categóricas. Ideal para comparar patrones entre grupos sin saturar un solo gráfico.

In [ ]:
# Demo: FacetGrid — edad por clase y supervivencia

g = sns.FacetGrid(
    titanic,
    col="pclass",  # una columna por cada clase
    hue="survived",  # color por supervivencia
    height=3.5,
    aspect=1
)

# Mapear un histograma en cada panel
g.map_dataframe(
    sns.histplot,
    x="age",
    kde=True,
    bins=15,
    stat="density"
)

g.add_legend(title="Survived")
g.fig.suptitle("Distribución de edades por clase del Titanic", y=1.01)
plt.tight_layout()

⏸️ **Pause and think:** ¿Notas diferencias en la edad promedio entre clases? ¿Y entre supervivientes y no-supervivientes dentro de cada clase?

### 🟡 Ejercicio 2 — Aplicado · FacetGrid con tips

⏱️ ~10 min · Mismo patrón, contexto nuevo

Usa el dataset `tips` (mesadas de restaurante) y crea un FacetGrid con:
- Una columna por `day` (día de la semana).
- Color por `sex` (sexo del pagador).
- Gráfico: scatter de `total_bill` vs `tip`.

**Output esperado:** 4 paneles (Thurs, Fri, Sat, Sun) con scatter points coloreados por sexo.

In [ ]:
# 🟡 Crea el FacetGrid

tips = sns.load_dataset("tips")

g = sns.FacetGrid(
    tips,
    col="___",  # ← por día
    hue="___",  # ← por sexo
    height=3,
    col_order=["Thurs", "Fri", "Sat", "Sun"]
)

g.map_dataframe(sns.scatterplot, x="total_bill", y="tip", s=80, alpha=0.6)
g.add_legend(title="Sex")
g.fig.suptitle("Propina vs Total por Día de la Semana", y=1.01)
plt.tight_layout()

In [ ]:
# Validación
assert "g" in dir(), "El objeto FacetGrid no existe"
assert g.fig is not None
print("✅ Ejercicio 2 completado")

In [ ]:
# %% [solution]
# g = sns.FacetGrid(
#     tips,
#     col="day",
#     hue="sex",
#     height=3,
#     col_order=["Thurs", "Fri", "Sat", "Sun"]
# )
# g.map_dataframe(sns.scatterplot, x="total_bill", y="tip", s=80, alpha=0.6)
# g.add_legend(title="Sex")
# g.fig.suptitle("Propina vs Total por Día de la Semana", y=1.01)
# plt.tight_layout()

---

## 3. Plotly Express · Interactividad sin fricción

⏱️ ~15 min · Gráficos interactivos

> 📌 **Concepto clave:** Plotly Express (`px`) es una API de alto nivel que convierte un DataFrame en gráficos web interactivos con una sola línea. Perfecto para dashboards y presentaciones donde el usuario puede explorar.

> 💡 **Tip:** En Colab, los gráficos de Plotly se muestran directamente. En Jupyter local, usa `fig.show()`. En Streamlit, con `st.plotly_chart(fig)`.

In [ ]:
# Demo: scatter interactivo

fig = px.scatter(
    titanic,
    x="age",
    y="fare",
    color="survived",
    size="pclass",  # tamaño proporcional a clase (clase 1 = más grande)
    hover_data=["name", "sex", "pclass"],  # info al pasar el mouse
    color_discrete_map={0: "#d62728", 1: "#2ca02c"},  # rojo=murió, verde=sobrevivió
    title="Titanic: Edad vs Tarifa (interactivo)",
    labels={"age": "Edad (años)", "fare": "Tarifa (£)", "survived": "Sobrevivió"}
)

fig.update_layout(height=500, font=dict(size=12))
fig.show()

Prueba hacer zoom, hover sobre puntos, y usa la leyenda para filtrar (click derecho en "0" o "1" para aislar grupos).

### 🟡 Ejercicio 3 — Aplicado · Bar plot interactivo

⏱️ ~8 min

Crea un gráfico de barras (bar plot) con Plotly que muestre **% de supervivencia por clase y sexo**. Usa `barmode="group"` para comparar lado a lado.

Pasos:
1. Agrupa por `pclass` y `sex`, calcula `survived.mean()` para obtener el porcentaje.
2. Resetea el índice con `.reset_index()`.
3. Usa `px.bar()` con `x="pclass"`, `y=[la columna de promedios]`, `color="sex"`, `barmode="group"`.

In [ ]:
# 🟡 Crea la agregación y el gráfico

# Paso 1: Agregar
agg_superviv = titanic.groupby(["pclass", "sex"])["survived"].mean().reset_index()
agg_superviv.columns = ["pclass", "sex", "survival_rate"]

# Paso 2: Crear el gráfico
fig_bar = px.___(  # ← completa
    agg_superviv,
    x="___",
    y="___",
    color="___",
    barmode="___",
    title="Tasa de supervivencia por clase y sexo",
    labels={"survival_rate": "Tasa de supervivencia", "pclass": "Clase", "sex": "Sexo"}
)

fig_bar.update_layout(height=500)
fig_bar.show()

In [ ]:
# Validación
assert "fig_bar" in dir()
assert agg_superviv.shape[0] > 0
print("✅ Ejercicio 3 completado")

In [ ]:
# %% [solution]
# fig_bar = px.bar(
#     agg_superviv,
#     x="pclass",
#     y="survival_rate",
#     color="sex",
#     barmode="group",
#     title="Tasa de supervivencia por clase y sexo",
#     labels={"survival_rate": "Tasa de supervivencia", "pclass": "Clase", "sex": "Sexo"}
# )
# fig_bar.update_layout(height=500)
# fig_bar.show()

---

## 4. Gráficos avanzados · jointplot y heatmap

⏱️ ~10 min

> 📌 **Concepto clave:** Un `jointplot` combina un scatter central con histogramas marginales (en los ejes) para analizar la distribución bivariada. Un `heatmap` de correlación muestra de un vistazo qué variables están relacionadas.

In [ ]:
# Demo: jointplot

fig = plt.figure(figsize=(8, 6))
jg = sns.jointplot(
    data=titanic.dropna(subset=["age", "fare"]),
    x="age",
    y="fare",
    kind="hex",  # hexbin: más hermoso que scatter cuando hay muchos puntos
    cmap="YlOrRd",
    height=6
)
jg.fig.suptitle("Edad vs Tarifa: Densidad hexagonal", y=1.0)
plt.tight_layout()

In [ ]:
# Demo: heatmap de correlación

# Seleccionar solo numéricas
numericas = titanic[["age", "fare", "pclass", "survived"]].dropna()
corr_matrix = numericas.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,  # mostrar valores
    cmap="coolwarm",  # azul=negativo, rojo=positivo
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Correlación"}
)
plt.title("Matriz de correlación · Titanic", fontsize=12, fontweight="bold")
plt.tight_layout()

### 🔴 Ejercicio 4 — Desafío · Dashboard exploratorio

⏱️ ~15-20 min · Open-ended

Construye un **análisis visual completo** del dataset `diamonds` (precios de diamantes). El análisis debe incluir:

1. **Un `pairplot`** con al menos 4 variables numéricas, coloreado por `cut` (calidad de corte).
2. **Un `heatmap` de correlación** de las variables numéricas.
3. **Un gráfico Plotly Express** (`scatter`, `box`, o `violin`) que muestre cómo el `price` varía según `cut`, `color` o `clarity`.

Cada gráfico debe tener:
- Título descriptivo (que resume el hallazgo, no solo el nombre de variables).
- Labels claros en ejes.
- Paleta de colores deliberada (no por defecto).

**Criterios de aceptación:**
- ✓ 3 visualizaciones distintas
- ✓ Código limpio y bien comentado
- ✓ Hallazgos escritos en prosa (3-5 oraciones) debajo de cada gráfico

In [ ]:
# 🔴 Tu turno — Completa el análisis

diamonds = sns.load_dataset("diamonds")
print(f"Shape: {diamonds.shape}")
print(f"Columnas: {diamonds.columns.tolist()}")
diamonds.head()

In [ ]:
# Visualización 1: Pairplot
# (escoge 4 variables numéricas y colorea por cut)

# ← Escribe tu código aquí

**Hallazgos del pairplot:**

← Escribe aquí 2-3 oraciones explicando qué ves. Ejemplo: "El precio correlaciona fuertemente con quilate (carat). Las piedras de mejor corte tienden a ser más pequeñas pero más caras por quilate."

In [ ]:
# Visualización 2: Heatmap de correlación

# ← Escribe tu código aquí

**Hallazgos del heatmap:**

← ¿Cuál es la correlación más fuerte? ¿Y la más sorprendente (si la hay)?

In [ ]:
# Visualización 3: Gráfico Plotly interactivo
# (elige uno: scatter, box, violin, bar, etc.)

# ← Escribe tu código aquí

**Hallazgos del gráfico interactivo:**

← ¿Qué insights sacas? ¿Hay outliers? ¿Grupos claros?

In [ ]:
# Validación básica
assert diamonds.shape[0] > 0, "Los datos no se cargaron"
print("✅ Ejercicio 4 entregado (revisión manual recomendada)")

---

## 🧭 Checkpoint · Auto-evaluación

Antes de avanzar, responde mentalmente:

1. ¿Cuándo usarías un `pairplot` vs un `FacetGrid`?
   - a) Pairplot para muchas variables numéricas, FacetGrid para comparar un gráfico entre grupos.
   - b) Son idénticos ← ✗
   - c) Depende del día de la semana ← ✗

2. ¿Qué hace `hue="survived"` en Seaborn?
   - a) Colorea los puntos/barras según la variable `survived` ← ✓
   - b) Añade una leyenda automáticamente (a veces) ← ✓
   - c) Ambas

3. ¿Cuál es la ventaja de Plotly Express sobre matplotlib/seaborn?
   - a) Interactividad: zoom, hover, selección.
   - b) Fácil de incrustar en Streamlit/dashboards.
   - c) Ambas ← ✓

📊 Has completado **4 de 4 secciones principales** · ¡Casi listo!

In [ ]:
# Validación de checkpoint
assert titanic.shape[0] == 891, "Reinicia y vuelve a cargar datos"
assert "pp" in dir() or "g" in dir() or "fig_bar" in dir(), "Completa los ejercicios"
print("Checkpoint ✓ — puedes continuar")

---

## 🧠 Mental model de hoy

```
Pregunta analítica
      ↓
  ¿Cuántas variables?
   ↙         ↓         ↘
2-3 vars   4+ vars   Temporal/Geo
   ↓          ↓           ↓
Scatter    Pairplot   Timeline/Map
o Box      FacetGrid    Plotly Geo
   ↓          ↓           ↓
Plotly    Seaborn    Plotly Express
Express   + Plotly
   ↓          ↓           ↓
  [Gráfico → Interpretación → Storytelling]
```

## 📌 Resumen · Lo que aprendiste hoy

- **Pairplot:** Matriz de correlaciones visualizada rápidamente, ideal para EDA inicial.
- **FacetGrid:** Divide un gráfico en paneles por grupo, perfecto para comparar patrones.
- **Plotly Express:** Gráficos interactivos con código limpio, integrables en dashboards.
- **Heatmap de correlación:** Identifica relaciones bivariadas de un vistazo.
- **Regla de oro:** Cada gráfico responde UNA pregunta clara, con título que resuma el hallazgo.

## 🚀 Próximos pasos (D+1)

Mañana veremos **hipótesis estadísticas y pruebas de significancia**. Para preparar:
- Relee la sección de "Atributos pre-atentivos" en el HTML si quieres reforzar diseño visual.
- Practica creando un FacetGrid con tu propio dataset (si tienes uno para el proyecto).

## 📚 Para profundizar

- [Seaborn docs · pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html)
- [Plotly Express · API Reference](https://plotly.com/python/plotly-express/)
- [Data to Viz: Chart Chooser](https://www.data-to-viz.com/) — la mejor guía para elegir gráfico según pregunta

## ✍️ Reflexión final

> En tus propias palabras, explícale a un compañero **cuándo y por qué usarías Plotly en vez de Seaborn** en máximo 3 oraciones. Considera contexto (presentación vs notebook), audiencia (técnica vs ejecutiva) e interactividad.

---

## 🧑‍🤝‍🧑 Trabajo en grupo — avance del proyecto

- **Hito 1:** Crea un pairplot o FacetGrid de tu dataset del proyecto. Guárdalo en el repo.
- **Hito 2:** Genera un heatmap de correlación e identifica las 3 pares de variables más correlacionadas.
- **Hito 3:** Diseña 1 gráfico Plotly interactivo que pueda ir al dashboard final.
- **Entrega:** Un notebook `eda_visualizations.ipynb` con los 3 gráficos + interpretación escrita (5-7 oraciones por gráfico).